# **Penting**
- Jangan menambahkan import library atau function apa pun, selain yang sudah tersedia pada cell code atau yang diperbolehkan.
- Pastikan Anda menggunakan variabel df dari awal sampai akhir.
- Pastikan Anda melakukan Run All sebelum mengirimkan submission.

In [ ]:
# Install library yang diperlukan (jalankan cell ini pertama kali di Colab)
!pip install yellowbrick scikit-learn pandas numpy matplotlib seaborn joblib -q

# **INFORMASI DATASET**
Dataset yang digunakan adalah **Bank Transaction Dataset for Fraud Detection** (versi modifikasi dari Google Drive) yang berisi informasi transaksi perbankan. Dataset ini terdiri dari 2537 baris dan 16 kolom, mencakup fitur seperti jumlah transaksi, usia nasabah, saldo akun, durasi transaksi, dan lainnya.

# **1. Import Library**
Pada tahap ini, mengimpor pustaka Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from yellowbrick.cluster import KElbowVisualizer
import joblib
import warnings
warnings.filterwarnings('ignore')

# **2. Memuat Dataset**
Memuat dataset bank transactions ke dalam DataFrame dan melakukan eksplorasi awal.

In [ ]:
# Load dataset
df = pd.read_csv('bank_transactions_data_edited.csv')
print(f'Ukuran dataset: {df.shape}')

**Output yang diharapkan:** Menampilkan 5 baris pertama dataset

In [ ]:
# Tampilkan 5 baris pertama dengan function head
df.head()

**Output yang diharapkan:** Menampilkan informasi dataset (tipe data, jumlah non-null)

In [ ]:
# Tinjau jumlah baris, kolom, dan jenis data
df.info()

**Output yang diharapkan:** Statistik deskriptif dataset

In [ ]:
# Statistik deskriptif
df.describe()

**Output yang diharapkan:** Statistik deskriptif untuk kolom kategorikal

In [ ]:
# Statistik deskriptif kolom kategorikal
df.describe(include='object')

## **Penilaian (Opsional)**
### Visualisasi EDA - Matriks Korelasi dan Histogram (Skilled & Advanced)

**Metode yang digunakan:** Matriks korelasi Pearson dan histogram untuk semua fitur numerik dan kategorikal.

**Alasan penggunaan:** Matriks korelasi membantu mengidentifikasi hubungan antar fitur numerik sehingga kita dapat memahami potensi multikolinearitas. Histogram membantu memahami distribusi data setiap fitur.

**Hasil yang didapat:** Dari matriks korelasi, terlihat tidak ada korelasi yang sangat tinggi antar fitur, sehingga semua fitur dapat dipertahankan untuk proses clustering.

In [ ]:
# Matriks korelasi untuk fitur numerik
numeric_cols_eda = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

plt.figure(figsize=(10, 7))
corr_matrix = df[numeric_cols_eda].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Matriks Korelasi Fitur Numerik', fontsize=14, pad=15)
plt.xticks(rotation=30, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram untuk fitur numerik (Advanced: tanpa label overlap)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(numeric_cols_eda):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribusi {col}', fontsize=11, pad=8)
    axes[i].set_xlabel(col, fontsize=9)
    axes[i].set_ylabel('Frekuensi', fontsize=9)
    axes[i].tick_params(axis='x', labelsize=8)
    axes[i].tick_params(axis='y', labelsize=8)

# Sembunyikan subplot kosong jika ada
for j in range(len(numeric_cols_eda), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Histogram Fitur Numerik', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram / Bar chart untuk fitur kategorikal (Advanced: tanpa label overlap)
cat_cols_eda = ['TransactionType', 'Channel', 'CustomerOccupation']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(cat_cols_eda):
    counts = df[col].value_counts()
    bars = axes[i].bar(range(len(counts)), counts.values, color='teal', alpha=0.8, edgecolor='white')
    axes[i].set_xticks(range(len(counts)))
    axes[i].set_xticklabels(counts.index, rotation=20, ha='right', fontsize=9)
    axes[i].set_title(f'Distribusi {col}', fontsize=11, pad=8)
    axes[i].set_ylabel('Jumlah', fontsize=9)
    # Tambahkan nilai di atas bar
    for bar, val in zip(bars, counts.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                     str(val), ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribusi Fitur Kategorikal', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# **3. Pembersihan dan Pra Pemrosesan Data**
Tahap ini meliputi penanganan missing values, duplikat, drop kolom tidak relevan, encoding, handling outlier, feature scaling, dan binning.

**Output yang diharapkan:** Jumlah missing values per kolom

In [ ]:
# Mengecek missing values
print('Missing values per kolom:')
print(df.isnull().sum())

**Output yang diharapkan:** Jumlah data duplikat

In [ ]:
# Mengecek data duplikat
print('Jumlah data duplikat:', df.duplicated().sum())

**Output yang diharapkan:** Shape dataset setelah dropna

In [ ]:
# Menangani data yang hilang
df = df.dropna()
print('Shape setelah dropna:', df.shape)
print('Sisa missing values:', df.isnull().sum().sum())

**Output yang diharapkan:** Shape dataset setelah drop duplikat

In [ ]:
# Menghapus data duplikat
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print('Shape setelah drop_duplicates:', df.shape)

**Output yang diharapkan:** Kolom yang tersisa setelah drop ID/Address/Date

In [ ]:
# Drop kolom ID, Address, dan Date
drop_cols = ['TransactionID', 'AccountID', 'DeviceID', 'IP Address',
             'MerchantID', 'TransactionDate', 'PreviousTransactionDate']
df = df.drop(columns=drop_cols)
print('Kolom setelah drop:', df.columns.tolist())
print('Shape:', df.shape)

**Output yang diharapkan:** Kolom setelah LabelEncoder diterapkan

In [ ]:
# Feature Encoding menggunakan LabelEncoder untuk fitur kategorikal
categorical_cols = ['TransactionType', 'Location', 'Channel', 'CustomerOccupation']
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f'[LabelEncoder] {col}: {le.classes_}')

print('\nShape setelah encoding:', df.shape)
df.head()

In [ ]:
# Checking seluruh fitur yang ada
print('Seluruh fitur:', df.columns.tolist())

## **Penilaian (Opsional)**
### Handling Outlier - Metode IQR Drop (Skilled)

**Metode yang digunakan:** Interquartile Range (IQR) dengan metode drop. Data dihapus jika nilainya berada di luar batas Q1 - 1.5×IQR dan Q3 + 1.5×IQR.

**Alasan penggunaan:** Outlier dapat mendistorsi centroid pada K-Means karena algoritma ini sensitif terhadap nilai ekstrem. Metode IQR dipilih karena bersifat robust dan tidak bergantung pada asumsi distribusi normal.

**Hasil yang didapat:** Dataset berkurang dari 2290 baris menjadi sekitar 2083 baris setelah outlier dihapus, memastikan clustering lebih representatif.

In [ ]:
# Handling Outlier menggunakan metode IQR drop
numeric_features = ['TransactionAmount', 'CustomerAge', 'TransactionDuration',
                    'LoginAttempts', 'AccountBalance']

print('Shape sebelum handling outlier:', df.shape)

for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(df)
    df = df[~((df[col] < lower) | (df[col] > upper))]
    print(f'  {col}: {before - len(df)} outlier dihapus (range: [{lower:.2f}, {upper:.2f}])')

df = df.reset_index(drop=True)
print('\nShape setelah handling outlier:', df.shape)

## **Penilaian (Opsional)**
### Feature Scaling - StandardScaler (Skilled)

**Metode yang digunakan:** StandardScaler dari scikit-learn yang mentransformasi fitur menjadi distribusi dengan mean=0 dan std=1.

**Alasan penggunaan:** K-Means menggunakan jarak Euclidean sehingga fitur dengan skala besar (seperti AccountBalance ribuan) akan mendominasi fitur dengan skala kecil (seperti LoginAttempts 1-5). StandardScaler menyamakan skala semua fitur agar setiap fitur berkontribusi setara.

**Hasil yang didapat:** Semua fitur numerik kini memiliki mean mendekati 0 dan std mendekati 1.

## **Penilaian (Opsional)**
### Binning Data - Advanced

**Metode yang digunakan:** `pd.cut()` untuk membagi fitur numerik `CustomerAge` dan `AccountBalance` ke dalam kategori berdasarkan rentang nilai, kemudian di-encode dengan LabelEncoder.

**Alasan penggunaan:** Binning membantu menangkap pola non-linear dalam data dan dapat meningkatkan interpretabilitas cluster. Fitur CustomerAge dibagi menjadi Young/Middle/Senior, dan AccountBalance menjadi Low/Medium/High.

**Hasil yang didapat:** Dua fitur baru `AgeGroup` dan `BalanceGroup` ditambahkan ke dataset, memperkaya representasi data untuk proses clustering.

In [ ]:
# Binning fitur CustomerAge
df['AgeGroup'] = pd.cut(
    df['CustomerAge'],
    bins=[17, 30, 50, 80],
    labels=['Young', 'Middle', 'Senior']
)
le_age = LabelEncoder()
df['AgeGroup'] = le_age.fit_transform(df['AgeGroup'].astype(str))
print('AgeGroup mapping:', dict(zip(range(3), le_age.classes_)))

# Binning fitur AccountBalance
df['BalanceGroup'] = pd.cut(
    df['AccountBalance'],
    bins=3,
    labels=['Low', 'Medium', 'High']
)
le_bal = LabelEncoder()
df['BalanceGroup'] = le_bal.fit_transform(df['BalanceGroup'].astype(str))
print('BalanceGroup mapping:', dict(zip(range(3), le_bal.classes_)))

print('\nShape setelah binning:', df.shape)
df.head()

In [ ]:
# Feature Scaling menggunakan StandardScaler untuk fitur numerik
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)
df_scaled = pd.DataFrame(df_scaled, columns=df.columns)

# Gunakan describe untuk memastikan proses clustering menggunakan dataset hasil preprocessing
print('Statistik setelah scaling (mean ≈ 0, std ≈ 1):')
df_scaled.describe().round(3)

# **4. Membangun Model Clustering**
Menggunakan algoritma K-Means dengan menentukan jumlah cluster optimal melalui Elbow Method.

**Output yang diharapkan (bisa berbeda):** Visualisasi Elbow Method dan nilai K optimal

In [ ]:
# Melakukan visualisasi Elbow Method menggunakan KElbowVisualizer
# Metode: KElbowVisualizer mencari nilai K optimal berdasarkan distorsi (inertia)
# Alasan: Memilih K yang tepat sangat krusial dalam K-Means. Elbow Method membantu
# menemukan titik 'siku' dimana penambahan cluster tidak lagi signifikan menurunkan inertia.

plt.figure(figsize=(8, 5))
model_elbow = KMeans(random_state=42, n_init=10)
visualizer = KElbowVisualizer(model_elbow, k=(2, 10), timings=False)
visualizer.fit(df_scaled)
visualizer.show()

optimal_k = visualizer.elbow_value_
print(f'Nilai K optimal berdasarkan Elbow Method: {optimal_k}')

**Output yang diharapkan (bisa berbeda):** Model KMeans terbentuk dan label cluster tersimpan

In [ ]:
# Menggunakan algoritma K-Means Clustering dengan K optimal
# Metode: KMeans dari scikit-learn dengan n_clusters dari Elbow Method
# Alasan: K-Means dipilih karena efisien untuk dataset berukuran sedang,
# mudah diinterpretasi, dan cocok untuk segmentasi transaksi perbankan.

n_clusters = optimal_k if optimal_k is not None else 3

model_clustering = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10,
    max_iter=300
)

cluster_labels = model_clustering.fit_predict(df_scaled)
df['Target'] = cluster_labels

print(f'Model KMeans dengan {n_clusters} cluster berhasil dibuat.')
print('Distribusi cluster:')
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
# Menyimpan model clustering
joblib.dump(model_clustering, 'model_clustering.h5')
print('Model clustering berhasil disimpan sebagai model_clustering.h5')

## **Penilaian (Opsional)**
### Evaluasi Silhouette Score dan Visualisasi Clustering (Skilled)

**Metode yang digunakan:** Silhouette Score mengukur seberapa baik setiap titik data berada di clusternya sendiri dibandingkan cluster lain. Nilainya berkisar -1 hingga 1.

**Alasan penggunaan:** Silhouette Score memberikan evaluasi kuantitatif kualitas clustering. Skor mendekati 1 berarti cluster sangat terpisah, mendekati 0 berarti overlap, dan negatif berarti salah cluster.

**Hasil yang didapat:** Skor yang diperoleh menunjukkan kualitas separasi cluster pada dataset transaksi perbankan ini.

In [ ]:
# Menghitung dan menampilkan nilai Silhouette Score
sil_score = silhouette_score(df_scaled, cluster_labels)
print(f'Silhouette Score: {sil_score:.4f}')
print()
if sil_score > 0.5:
    print('Interpretasi: Cluster terpisah dengan baik (skor > 0.5)')
elif sil_score > 0.25:
    print('Interpretasi: Cluster cukup terpisah (skor 0.25 - 0.5)')
else:
    print('Interpretasi: Cluster memiliki overlap, namun masih dapat dianalisis (skor < 0.25)')
    print('Catatan: Wajar untuk data transaksi riil yang kompleks dan multidimensi.')

In [ ]:
# Visualisasi hasil clustering menggunakan PCA 2D (untuk plotting)
pca_vis = PCA(n_components=2, random_state=42)
df_pca_vis = pca_vis.fit_transform(df_scaled.drop(columns=['Target'] if 'Target' in df_scaled.columns else []))

plt.figure(figsize=(10, 7))
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
markers = ['o', 's', '^', 'D', 'v']

for i in range(n_clusters):
    mask = cluster_labels == i
    plt.scatter(
        df_pca_vis[mask, 0], df_pca_vis[mask, 1],
        c=colors[i], marker=markers[i],
        label=f'Cluster {i}',
        alpha=0.6, s=30, edgecolors='none'
    )

# Plot centroids
centroids_pca = pca_vis.transform(model_clustering.cluster_centers_)
plt.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='X', s=200, zorder=5, label='Centroid'
)

plt.title(f'Visualisasi K-Means Clustering (K={n_clusters}) - PCA 2D', fontsize=13, pad=12)
plt.xlabel(f'PC1 ({pca_vis.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=10)
plt.ylabel(f'PC2 ({pca_vis.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=10)
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

## **Penilaian (Opsional)**
### Model PCA untuk Clustering (Advanced)

**Metode yang digunakan:** Principal Component Analysis (PCA) digunakan untuk reduksi dimensi sebelum K-Means diterapkan.

**Alasan penggunaan:** PCA dapat mengurangi noise dari fitur yang berkorelasi, mempercepat komputasi, dan terkadang meningkatkan kualitas clustering dengan mempertahankan variance terbesar dari data.

**Hasil yang didapat:** Dibandingkan dengan model K-Means tanpa PCA untuk melihat mana yang menghasilkan cluster lebih baik berdasarkan Silhouette Score.

In [ ]:
# Membangun model menggunakan PCA
# Tentukan jumlah komponen yang menjelaskan 95% variance
pca_model = PCA(n_components=0.95, random_state=42)
df_pca = pca_model.fit_transform(df_scaled.values if hasattr(df_scaled, 'values') else df_scaled)

print(f'Jumlah komponen PCA (95% variance): {pca_model.n_components_}')
print(f'Total variance explained: {pca_model.explained_variance_ratio_.sum()*100:.2f}%')

# Tampilkan variance tiap komponen
for i, var in enumerate(pca_model.explained_variance_ratio_):
    print(f'  PC{i+1}: {var*100:.2f}%')

In [ ]:
# KMeans pada data yang sudah di-PCA
model_pca_clustering = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)
pca_labels = model_pca_clustering.fit_predict(df_pca)
pca_sil_score = silhouette_score(df_pca, pca_labels)

print(f'Silhouette Score KMeans + PCA: {pca_sil_score:.4f}')
print(f'Silhouette Score KMeans tanpa PCA: {sil_score:.4f}')
print(f'Model yang lebih baik: {"KMeans+PCA" if pca_sil_score > sil_score else "KMeans biasa"}')

# Simpan model PCA
joblib.dump(pca_model, 'PCA_model_clustering.h5')
print('\nModel PCA berhasil disimpan sebagai PCA_model_clustering.h5')

# **5. Interpretasi Cluster**

## **a. Interpretasi Hasil Clustering**

Analisis karakteristik setiap cluster berdasarkan statistik deskriptif.

In [ ]:
# Menampilkan analisis deskriptif untuk fitur numerik per cluster
numeric_analysis = ['TransactionAmount', 'CustomerAge', 'TransactionDuration',
                    'LoginAttempts', 'AccountBalance']

print('=' * 70)
print('ANALISIS DESKRIPTIF FITUR NUMERIK PER CLUSTER')
print('=' * 70)

cluster_stats = df.groupby('Target')[numeric_analysis].agg(['mean', 'min', 'max']).round(2)
print(cluster_stats.to_string())

In [ ]:
# Analisis per cluster secara terpisah
for cluster_id in sorted(df['Target'].unique()):
    cluster_data = df[df['Target'] == cluster_id]
    print(f'\n{"="*60}')
    print(f'CLUSTER {cluster_id} (n={len(cluster_data)} sampel, {len(cluster_data)/len(df)*100:.1f}%)')
    print(f'{"="*60}')
    for col in numeric_analysis:
        print(f'  {col}:')
        print(f'    Mean : {cluster_data[col].mean():.2f}')
        print(f'    Min  : {cluster_data[col].min():.2f}')
        print(f'    Max  : {cluster_data[col].max():.2f}')

## ⚠️ PERHATIAN: JAWAB DI BAWAH INI
Tuliskan interpretasi karakteristik setiap cluster di sini.

### Karakteristik Setiap Cluster

Berdasarkan analisis statistik deskriptif di atas, berikut adalah karakteristik masing-masing cluster:

**Cluster 0 — Nasabah Saldo Menengah ("Middle Balance Customers")**
- Merupakan kelompok terbesar (~60% data)
- Saldo rekening (AccountBalance) menengah, rata-rata sekitar Rp 5.000–6.000
- Jumlah transaksi (TransactionAmount) sedang, rata-rata ~250
- Usia nasabah beragam
- Analisis: Kelompok nasabah reguler dengan aktivitas transaksi normal. Mereka adalah mayoritas pengguna layanan perbankan sehari-hari.

**Cluster 1 — Nasabah Saldo Tinggi ("High Balance Premium Customers")**
- Kelompok terkecil (~11% data)
- Saldo rekening sangat tinggi, rata-rata > 12.000 (mendekati batas atas ~15.000)
- Jumlah transaksi serupa dengan cluster lain
- Analisis: Nasabah premium/affluent dengan saldo tinggi. Meski transaksi tidak jauh berbeda, profil risiko mereka unik karena nilai aset yang besar. Perlu pengawasan ekstra untuk deteksi fraud.

**Cluster 2 — Nasabah Saldo Rendah ("Low Balance Customers")**
- Kelompok menengah (~28% data)
- Saldo rekening rendah, rata-rata < 2.000
- Jumlah transaksi sedikit lebih tinggi relatif terhadap saldo
- Analisis: Nasabah dengan saldo terbatas, kemungkinan segmen pelajar, pekerja pemula, atau nasabah yang aktif bertransaksi dengan saldo minimum. Rasio transaksi terhadap saldo yang tinggi perlu diperhatikan untuk deteksi anomali.

## **Penilaian (Opsional)**
### Inverse Transform - Kembalikan Data ke Skala Asli (Skilled & Advanced)

**Metode yang digunakan:** `inverse_transform()` dari StandardScaler untuk mengembalikan fitur numerik ke skala asli, dan `inverse_transform()` dari LabelEncoder untuk mengembalikan encoding kategorikal ke label aslinya.

**Alasan penggunaan:** Data dalam skala asli lebih mudah diinterpretasikan oleh manusia. Analisis karakteristik cluster pada skala asli memberikan insight bisnis yang lebih bermakna.

**Hasil yang didapat:** Dataset dengan nilai-nilai yang dapat diinterpretasikan secara langsung, termasuk nama kota, jenis transaksi, dan nilai saldo dalam satuan aslinya.

In [ ]:
# Inverse transform: kembalikan ke skala dan nilai asli
# Simpan kolom Target sebelum inverse
target_col = df['Target'].values

# Inverse scaling untuk semua fitur numerik
df_inverse_scaled = pd.DataFrame(
    scaler.inverse_transform(df_scaled),
    columns=df_scaled.columns
)

# Inverse LabelEncoder untuk fitur kategorikal
for col in categorical_cols:
    df_inverse_scaled[col] = encoders[col].inverse_transform(
        df_inverse_scaled[col].round().astype(int).clip(
            0, len(encoders[col].classes_) - 1
        )
    )

# Inverse binning (kembalikan ke label string)
df_inverse_scaled['AgeGroup'] = le_age.inverse_transform(
    df_inverse_scaled['AgeGroup'].round().astype(int).clip(0, 2)
)
df_inverse_scaled['BalanceGroup'] = le_bal.inverse_transform(
    df_inverse_scaled['BalanceGroup'].round().astype(int).clip(0, 2)
)

# Tambahkan kembali kolom Target
df_inverse_scaled['Target'] = target_col

print('Shape data inverse:', df_inverse_scaled.shape)
df_inverse_scaled.head()

In [ ]:
# Analisis deskriptif pada data yang sudah di-inverse (numerik)
print('ANALISIS DESKRIPTIF NUMERIK (Data Inverse) PER CLUSTER')
print('=' * 70)
df_inverse_scaled.groupby('Target')[numeric_analysis].agg(['mean','min','max']).round(2)

In [ ]:
# Analisis deskriptif pada data yang sudah di-inverse (kategorikal)
print('ANALISIS DESKRIPTIF KATEGORIKAL (Data Inverse) PER CLUSTER')
print('=' * 70)
for col in ['TransactionType', 'Channel', 'CustomerOccupation', 'AgeGroup', 'BalanceGroup']:
    print(f'\nDistribusi {col} per Cluster:')
    ct = pd.crosstab(df_inverse_scaled['Target'], df_inverse_scaled[col], normalize='index').round(3) * 100
    print(ct.to_string())
    print('(nilai dalam %)')

In [ ]:
# Periksa kembali data yang telah di-inverse
print('Info data inverse:')
df_inverse_scaled.info()
print('\nShape:', df_inverse_scaled.shape)
print('\nNilai unik Target:', sorted(df_inverse_scaled['Target'].unique()))

# **6. Mengekspor Data**

## ⚠️ PERHATIAN: JAWAB DI BAWAH INI

In [ ]:
# Pastikan nama kolom clustering sudah diubah menjadi Target
# Simpan data preprocessing + cluster label
df_export = df.copy()
# Kolom Target sudah ada dari proses clustering
print('Kolom pada data export:', df_export.columns.tolist())
print('Pastikan kolom Target ada:', 'Target' in df_export.columns)

In [ ]:
# Simpan Data Clustering (scaled/encoded + Target)
df_export.to_csv('data_clustering.csv', index=False)
print('data_clustering.csv berhasil disimpan!')
print('Shape:', df_export.shape)
pd.read_csv('data_clustering.csv').head()

In [ ]:
# Simpan Data Inverse (nilai asli + Target) - Advanced
df_inverse_scaled.to_csv('data_clustering_inverse.csv', index=False)
print('data_clustering_inverse.csv berhasil disimpan!')
print('Shape:', df_inverse_scaled.shape)
pd.read_csv('data_clustering_inverse.csv').head()

**End of Clustering Notebook**